# 02 - QLoRA fine-tuning with Unsloth (Colab T4)

Base model: `Qwen/Qwen2.5-Coder-3B-Instruct`. Dataset: ChatML JSONL from notebook 01.

## Hardware

- **Target: Google Colab with a T4 (15 GB, Ampere, native bf16)** — the free tier is enough for a 3B QLoRA run.
- Runtime: `Runtime > Change runtime type > T4 GPU`.
- Local GTX 1050 Ti (4 GB) is **not** suitable for training, only for inference via Ollama after export.

In [ ]:
# Sanity-check the GPU assigned by Colab.
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
# %%capture
!pip install -U pip
!pip install "unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27">2.0.1 trl peft accelerate bitsandbytes

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=2048,          # T4-friendly; raise to 4096 only if VRAM allows
    dtype=torch.bfloat16,         # native on T4/Ampere
    load_in_4bit=True,            # QLoRA: 4-bit NF4 base weights
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
print(model.print_trainable_parameters())

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_json("data/chatml/train.jsonl")
eval_ds  = Dataset.from_json("data/chatml/val.jsonl")  # optional 5% split by db_id

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=8,
    args=SFTConfig(
        per_device_train_batch_size=4,    # 4 * 8 grad-accum = 32 effective batch
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_ratio=0.08,
        num_train_epochs=2,
        logging_steps=10,
        max_steps=-1,
        output_dir="checkpoints/zeroerr-3b-lora",
        save_strategy="epoch",
    ),
)

trainer_stats = trainer.train()

Next: run `notebooks/03_gguf_export.ipynb` (same session) to save the merged 16-bit checkpoint and a Q4_K_M GGUF, then download it to your machine for Ollama.